In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

project_root = Path.cwd().parent.resolve()
sys.path.insert(0, str(project_root))

# Avoid collisions with similarly named third-party modules in notebook contexts
if "data" in sys.modules:
    del sys.modules["data"]

from data.load import load_dataset, load_config
from data.preprocess import preprocess_train, preprocess_test
from features.health_index import build_health_index
from features.velocity import build_velocity
from features.variability import build_variability
from model.clustering import build_clustering
from model.risk import build_risk_score
from model.rul import build_rul_model

config = load_config(project_root / "config" / "config.yaml")
config["dataset"]["raw_path"] = str(project_root / "data" / "raw")

train_raw, test_raw, rul_raw = load_dataset(config)
train_proc, scaler, sensor_cols = preprocess_train(train_raw, config)
test_proc = preprocess_test(test_raw, config, scaler)

# Reconstruct cycle-level test RUL using provided end-of-trajectory offsets
test_max_cycle = test_proc.groupby("unit")["cycle"].max().rename("max_cycle")
test_proc = test_proc.merge(test_max_cycle, on="unit", how="left")
test_proc["RUL"] = (test_proc["max_cycle"] - test_proc["cycle"]) + test_proc["unit"].map(rul_raw)
test_proc = test_proc.drop(columns=["max_cycle"])

train_hi, test_hi, hi_art = build_health_index(train_proc, test_proc, config)
train_vel, test_vel, vel_art = build_velocity(train_hi, test_hi, config)
train_var, test_var, var_art = build_variability(train_vel, test_vel, config)
train_cl, test_cl, cl_art = build_clustering(train_var, test_var, config)
train_rs, test_rs, risk_art = build_risk_score(train_cl, test_cl, cl_art)
preds_df, rul_art = build_rul_model(train_rs, test_rs, config)

reports_dir = project_root / "reports"
reports_dir.mkdir(exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("RUL Model Evaluation — Gradient Boosting (FD001)", fontsize=14, fontweight="bold")

ax1 = axes[0]
ax1.scatter(
    preds_df["true_RUL"],
    preds_df["predicted_RUL"],
    alpha=0.6,
    color="steelblue",
    edgecolors="white",
    linewidth=0.5,
    s=60,
 )
max_val = max(preds_df["true_RUL"].max(), preds_df["predicted_RUL"].max()) + 10
ax1.plot([0, max_val], [0, max_val], "r--", linewidth=1.5, label="Perfect prediction")
ax1.set_xlabel("True RUL (cycles)")
ax1.set_ylabel("Predicted RUL (cycles)")
ax1.set_title("Predicted vs True RUL")
ax1.legend()

ax2 = axes[1]
colors = ["tomato" if error >= 0 else "steelblue" for error in preds_df["error"]]
ax2.scatter(
    preds_df["true_RUL"],
    preds_df["error"],
    alpha=0.6,
    c=colors,
    edgecolors="white",
    linewidth=0.5,
    s=60,
 )
ax2.axhline(0, color="red", linestyle="--", linewidth=1.5)
ax2.set_xlabel("True RUL (cycles)")
ax2.set_ylabel("Error (predicted - true)")
ax2.set_title("Prediction Error vs True RUL\nRed = late (dangerous), Blue = early (safe)")

plt.tight_layout()
plt.savefig(reports_dir / "rul_evaluation_plots.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"RMSE       : {rul_art.evaluation_metrics['gradient_boosting']['rmse']:.2f} cycles")
print(f"NASA Score : {rul_art.evaluation_metrics['gradient_boosting']['nasa_score']:.1f}")
print(f"Late preds : {(preds_df['error'] > 0).sum()} engines")
print(f"Early preds: {(preds_df['error'] < 0).sum()} engines")

In [ ]:
print(preds_df.columns.tolist())
print(preds_df[["ci_lower", "ci_upper", "ci_std"]].head())
print(preds_df["ci_std"].describe())

In [ ]:
import importlib
import features.health_index as hi_module

hi_module = importlib.reload(hi_module)
from features.health_index import compute_sensor_contributions, build_health_index

# Rebuild HI artifacts from the reloaded module so loadings are available
train_hi, test_hi, hi_art = build_health_index(train_proc, test_proc, config)

# Test on engine 82 (the Critical one)
engine_82 = train_rs[train_rs["unit"] == 82].copy()
contrib = compute_sensor_contributions(engine_82, hi_art, sensor_cols)

print("Columns:", contrib.columns.tolist())
print("\nLast 5 cycles — top contributing sensor:")
print(contrib[["cycle", "top_sensor", "top_contribution_pct"]].tail())
print("\nMean absolute contribution per sensor (last 20 cycles):")
contrib_cols = [c for c in contrib.columns if c.endswith("_contribution")]
print(
    contrib[contrib_cols].tail(20).abs().mean()
    .sort_values(ascending=False)
    .head(5)
)

In [ ]:
from evaluation.validation import detect_anomalous_engines

anomalies = detect_anomalous_engines(train_rs)
print(f"Total engines: {len(anomalies)}")
print(f"Anomalous engines: {anomalies['is_anomaly'].sum()}")
print("\nAnomalous engines detail:")
print(anomalies[anomalies["is_anomaly"]][
    ["unit", "mahalanobis_distance", "anomaly_reason"]
])